In [ ]:
import numpy as np
from collections import Counter


class KNearstNeighbor:

    def train(self, x_train, y_train):
        self.x_train = x_train
        self.y_train = y_train
    
    def predict(self, x_data, k=1):
        nearst_neighbors = []
        for x in x_data:
            distances_between_points = self.__euclidean_distance(
                point_a=x,
                points_b=self.x_train,
                labels_b=self.y_train)

            # sort by distance and carry label within
            sorted_distances = sorted(
                distances_between_points, key=lambda t: t[0])
            nearst_neighbors.append(
                sorted_distances[:k]
            )

        return self.__vote(nearst_neighbors)

    def __vote(self, nearst_neighbors):
        predicts = []
        for neighbors in nearst_neighbors:
            label_counter = Counter([neighbor[1] for neighbor in neighbors])
            [(predicted_label, _)] = label_counter.most_common(1)
            predicts.append([predicted_label])
        return predicts

    def __euclidean_distance(self, point_a, points_b, labels_b):
        distances = []
        size_points_b = len(points_b)
        for i in range(size_points_b):
            point = points_b[i]
            point_label = labels_b[i][0]

            distance = np.sqrt(np.sum(np.square(
                    point_a - point)))
            distances.append(
                (distance, point_label))
        return distances

    def avaliar_desequilibrio(y_real, y_pred, classe_minoritaria):
        y_real = np.array(y_real).flatten()
        y_pred = np.array(y_pred).flatten()
        
        # Verdadeiros Positivos, Falsos Positivos, etc.
        TP = np.sum((y_pred == classe_minoritaria) & (y_real == classe_minoritaria))
        FP = np.sum((y_pred == classe_minoritaria) & (y_real != classe_minoritaria))
        FN = np.sum((y_pred != classe_minoritaria) & (y_real == classe_minoritaria))
        
        # Calcular métricas (com proteção contra divisão por zero)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = np.mean(y_pred == y_real)
        
        print(f"Accuracy:  {accuracy:.4f}")#(Enganadora em dados desequilibrados)
        print(f"Precision: {precision:.4f}")#(Quando diz que é anomalia, qual a % de acerto?)
        print(f"Recall:    {recall:.4f}")#(Das anomalias totais, quantas conseguiu apanhar?)
        print(f"F1-Score:  {f1_score:.4f}")#(O equilíbrio entre Precision e Recall)
        
        return accuracy, precision, recall, f1_score

In [ ]:
from matplotlib import pyplot as plt

#Falta importar o dataset
def see_some_samples(traing_x):
    for i in range(9):
         plt.subplot(330 + 1 + i)
         plt.imshow(traing_x[i])
    plt.show()

# include indexes in array to separate by classes 
#Label  Description
#0  airplane
#1  automobile
#2  bird
#3  cat
#4  deer
#5  dog
#6  frog
#7  horse
#8  ship
#9  truck
classes_to_use = [0, 1]


(traing_x, traing_y), (test_x, test_y) = cifar10.load_data() #aqui tem o dataset cifar10
classes_training = np.isin(traing_y, classes_to_use).flatten()
classes_testing = np.isin(test_y, classes_to_use).flatten()
traing_x = traing_x[classes_training]
traing_y = traing_y[classes_training]
test_x = test_x[classes_testing]
test_y = test_y[classes_testing]

# limit samples at 500 to avoid evalute to get slow
# as KNN is quite slow
SAMPLES_COUNT = 500
traing_x = traing_x[:SAMPLES_COUNT]
traing_y = traing_y[:SAMPLES_COUNT]

test_x = test_x[:SAMPLES_COUNT]
test_y = test_y[:SAMPLES_COUNT]
see_some_samples(traing_x)

In [ ]:
knn_classifier = KNearstNeighbor()
knn_classifier.train(
    x_train=traing_x,
    y_train=traing_y)
accuracy = knn_classifier.evaluate(test_x, test_y, k=5)

print(f'accuracy was {round(accuracy[0], 2) * 100} %')

In [ ]:
#versão com pesos nas classes minoritárias:

In [ ]:
import numpy as np
from collections import Counter

class CW_RBF_KNN:
    """
    Correlation-Weighted Radial Basis Function K-Nearest Neighbors
    Desenvolvido para lidar com Desequilíbrio de Classes (Class Imbalance).
    """
    
    def __init__(self, gamma=0.1):
        # Hiperparâmetro que controla a largura da "bolha" de influência do Kernel RBF
        self.gamma = gamma
        self.class_weights = {}
        self.feature_weights = []
        self.x_train = None
        self.y_train = None

    def train(self, x_train, y_train):
        # Converter para arrays numpy e achatar y_train (caso venha como lista de listas)
        self.x_train = np.array(x_train)
        y_flat = np.array(y_train).flatten()
        self.y_train = y_flat
        
        total_samples = len(y_flat)
        counts = Counter(y_flat)
        
        # 1. PESOS DAS CLASSES (Suavizados com Raiz Quadrada)
        # Dá mais poder de voto à classe minoritária
        self.class_weights = {label: np.sqrt(total_samples / count) for label, count in counts.items()}

        # 2. PESOS DAS VARIÁVEIS (Correlação Absoluta)
        # Binarizar temporariamente a target para o cálculo da correlação (0 e 1)
        unique_labels = list(counts.keys())
        y_numeric = np.where(y_flat == unique_labels[0], 0, 1)
        
        self.feature_weights = []
        num_features = self.x_train.shape[1]
        
        for i in range(num_features):
            coluna = self.x_train[:, i]
            
            # Se a coluna for constante, a correlação é impossível (desvio padrão 0)
            if np.std(coluna) == 0:
                self.feature_weights.append(0.0)
            else:
                corr = np.corrcoef(coluna, y_numeric)[0, 1]
                # Se a correlação der NaN por alguma anomalia matemática, assume 0
                if np.isnan(corr):
                    self.feature_weights.append(0.0)
                else:
                    self.feature_weights.append(np.abs(corr)) # Valor Absoluto!
                
        self.feature_weights = np.array(self.feature_weights)

    def predict(self, x_data, k=1):
        x_data = np.array(x_data)
        predicts = []
        
        for x in x_data:
            # 1. Calcular a Similaridade RBF em relação a todos os pontos de treino
            similarities_between_points = self.__rbf_similarity(x)

            # 2. Ordenar de forma DECRESCENTE (no RBF, maior = melhor/mais perto)
            sorted_similarities = sorted(similarities_between_points, key=lambda t: t[0], reverse=True)
            
            # 3. Escolher os K vizinhos mais "similares"
            k_nearest = sorted_similarities[:k]
            
            # 4. Fazer a votação final para este ponto
            predicted_label = self.__vote(k_nearest)
            predicts.append(predicted_label)

        return np.array(predicts)

    def __rbf_similarity(self, point_a):
        # Esta função tira partido de operações vetorizadas do NumPy (muito mais rápido que usar 'for')
        
        # 1. Diferença ao quadrado entre o ponto novo e TODOS os pontos de treino
        diferenca_quadrado = np.square(self.x_train - point_a)
        
        # 2. Multiplicar pela Correlação de cada variável e somar as linhas
        distancia_ponderada = np.sum(self.feature_weights * diferenca_quadrado, axis=1)
        
        # 3. Aplicar o Kernel RBF
        similarities = np.exp(-self.gamma * distancia_ponderada)
        
        # Juntar a similaridade de cada ponto à sua respetiva label
        return list(zip(similarities, self.y_train))

    def __vote(self, neighbors):
        class_scores = {}
        
        for similarity, label in neighbors:
            # A NOSSA FÓRMULA FINAL: Voto = Peso da Classe * Similaridade RBF
            vote_power = self.class_weights[label] * similarity
            class_scores[label] = class_scores.get(label, 0) + vote_power
            
        # Devolve a classe com a maior pontuação
        return max(class_scores, key=class_scores.get)

    def avaliar_desequilibrio(y_real, y_pred, classe_minoritaria):
        y_real = np.array(y_real).flatten()
        y_pred = np.array(y_pred).flatten()
        
        # Verdadeiros Positivos, Falsos Positivos, etc.
        TP = np.sum((y_pred == classe_minoritaria) & (y_real == classe_minoritaria))
        FP = np.sum((y_pred == classe_minoritaria) & (y_real != classe_minoritaria))
        FN = np.sum((y_pred != classe_minoritaria) & (y_real == classe_minoritaria))
        
        # Calcular métricas (com proteção contra divisão por zero)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = np.mean(y_pred == y_real)
        
        print(f"Accuracy:  {accuracy:.4f}")#(Enganadora em dados desequilibrados)
        print(f"Precision: {precision:.4f}")#(Quando diz que é anomalia, qual a % de acerto?)
        print(f"Recall:    {recall:.4f}")#(Das anomalias totais, quantas conseguiu apanhar?)
        print(f"F1-Score:  {f1_score:.4f}")#(O equilíbrio entre Precision e Recall)
        
        return accuracy, precision, recall, f1_score